# IFCE Campus Tauá - Tecnologia em Telemática
## Banco de Dados (2026.2) — Aula 06: Entidades Fracas e Modelo EER
**Prof. Me. Reginaldo Fernandes**

---
### Objetivos Práticos Deste Caderno:
1. **Entidades Fracas e Discriminadores:** Compreender a dependência existencial e de identificação entre uma entidade forte proprietária (`HOTEL`) e sua entidade fraca dependente (`QUARTO`).
2. **Chave Primária Composta por Dependência:** Demonstrar como a chave primária da entidade proprietária unida ao discriminador forma a identificação global unívoca `(id_hotel, numero_quarto)`.
3. **Modelo EER (Especialização/Generalização):** Modelar hierarquias de herança onde subclasses (`Carro`, `Caminhao`) herdam propriedades da superclasse (`Veiculo`).
4. **Restrições de Disjunção ($d$) e Integralidade Total:** Auditar e interceptar violações de regras semânticas em memória.

### 1. Definição das Classes para Entidades Fracas e Identificadores (Hotel e Quarto)

In [ ]:
from enum import Enum
from typing import Dict, Any, Optional, Set, List

class Hotel:
    """Entidade Forte: possui chave primária autônoma (id_hotel)."""
    def __init__(self, id_hotel: int, nome: str, cidade: str):
        self.id_hotel = id_hotel
        self.nome = nome
        self.cidade = cidade
        self.quartos_associados: Dict[str, 'Quarto'] = {}

    def __repr__(self) -> str:
        return f"<Hotel id={self.id_hotel} nome='{self.nome}'>"

class Quarto:
    """Entidade Fraca: identificada pelo discriminador numero_quarto + hotel proprietário."""
    def __init__(self, numero_quarto: str, tipo_leito: str, valor_diaria: float):
        self.numero_quarto = numero_quarto  # Chave Parcial / Discriminador
        self.tipo_leito = tipo_leito
        self.valor_diaria = valor_diaria
        self.hotel: Optional[Hotel] = None

    @property
    def chave_completa(self) -> Optional[tuple]:
        if not self.hotel:
            return None
        return (self.hotel.id_hotel, self.numero_quarto)

    def __repr__(self) -> str:
        chave = self.chave_completa or "SEM_HOTEL"
        return f"<Quarto {chave} ({self.tipo_leito}, R$ {self.valor_diaria:.2f})>"

class RelacionamentoIdentificador:
    """Garante integridade existencial e unicidade do discriminador."""
    def __init__(self, nome: str = "POSSUI_QUARTO"):
        self.nome = nome
        self.ligacoes: Dict[tuple, Quarto] = {}

    def vincular_quarto(self, hotel: Hotel, quarto: Quarto) -> None:
        if quarto.numero_quarto in hotel.quartos_associados:
            raise ValueError(
                f"[VIOLAÇÃO DE DISCRIMINADOR] O hotel '{hotel.nome}' "
                f"já possui o quarto número '{quarto.numero_quarto}' cadastrado!"
            )
        quarto.hotel = hotel
        hotel.quartos_associados[quarto.numero_quarto] = quarto
        self.ligacoes[quarto.chave_completa] = quarto
        print(f"✓ Quarto '{quarto.numero_quarto}' vinculado com sucesso ao '{hotel.nome}'.")

### 2. Executando o Teste de Entidade Fraca e Discriminador

In [ ]:
rel = RelacionamentoIdentificador()

h1 = Hotel(101, "Hotel Tauá Palace", "Tauá - CE")
h2 = Hotel(202, "Beira Mar Hotel", "Fortaleza - CE")

# Ambos os quartos possuem o mesmo discriminador '101' em hotéis diferentes
q1 = Quarto("101", "Casal Standard", 180.00)
q2 = Quarto("101", "Suíte Luxo", 350.00)

rel.vincular_quarto(h1, q1)
rel.vincular_quarto(h2, q2)

print(f"Chave Composta do Quarto 1: {q1.chave_completa}")
print(f"Chave Composta do Quarto 2: {q2.chave_completa}")

# Teste de bloqueio de discriminador duplicado no mesmo hotel
try:
    q_duplicado = Quarto("101", "Individual Simples", 120.00)
    rel.vincular_quarto(h1, q_duplicado)
except ValueError as erro:
    print(f"Bloqueio disparado com sucesso: {erro}")

### 3. Modelo EER: Superclasse Veículo, Subclasses Carro e Caminhão com Herança

In [ ]:
class Veiculo:
    def __init__(self, placa: str, marca: str, modelo: str, ano: int):
        self.placa = placa
        self.marca = marca
        self.modelo = modelo
        self.ano = ano

class Carro(Veiculo):
    def __init__(self, placa: str, marca: str, modelo: str, ano: int, quantidade_portas: int, capacidade_porta_malas_litros: int):
        super().__init__(placa, marca, modelo, ano)
        self.quantidade_portas = quantidade_portas
        self.capacidade_porta_malas_litros = capacidade_porta_malas_litros

class Caminhao(Veiculo):
    def __init__(self, placa: str, marca: str, modelo: str, ano: int, capacidade_carga_toneladas: float, quantidade_eixos: int):
        super().__init__(placa, marca, modelo, ano)
        self.capacidade_carga_toneladas = capacidade_carga_toneladas
        self.quantidade_eixos = quantidade_eixos

carro = Carro("POQ-7F89", "Toyota", "Corolla", 2024, 4, 470)
caminhao = Caminhao("KLY-9B34", "Volvo", "FH 540", 2023, 30.5, 6)

print("Herança em Carro:", vars(carro))
print("Herança em Caminhão:", vars(caminhao))